# DeepSets Extrapolation Training

Train DeepSets on various distance splits to study extrapolation.

**Splits:**
- `equal_333333`: Equal mix of d=3, d=5, d=7 (33% each)
- `no_d3`: Only d=5, d=7 (50% each)
- `d5_heavy`: 20% d=3, 60% d=5, 20% d=7
- `d7_only`: 100% d=7 (baseline)

**Uses best hyperparameters from tuning.**

In [ ]:
import sys
import json
import random
import time
from pathlib import Path
from datetime import datetime

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/deepsets/extrapolation -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
from tqdm.auto import tqdm

from benchmark_models import DeepSets

# Paths
EXTRAP_DIR = BASE_PATH / "deepsets" / "extrapolation"
MODELS_DIR = EXTRAP_DIR / "models"
RESULTS_DIR = EXTRAP_DIR / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Best Hyperparameters

In [ ]:
# Load best config from tuning
BEST_CONFIG_PATH = BASE_PATH / "deepsets" / "tuning" / "best_model_config.json"

if BEST_CONFIG_PATH.exists():
    with open(BEST_CONFIG_PATH, 'r') as f:
        best_config = json.load(f)
    print(f"Loaded best config from tuning:")
    print(f"  phi_hidden: {best_config['model_params']['phi_hidden']}")
    print(f"  rho_hidden: {best_config['model_params']['rho_hidden']}")
    print(f"  pool: {best_config['model_params']['pool']}")
    print(f"  lr: {best_config['training_params']['learning_rate']}")
else:
    print("No best config found. Using defaults.")
    best_config = {
        'model_params': {
            'phi_hidden': (128, 128),
            'rho_hidden': (256, 128),
            'pool': 'sum',
            'dropout': 0.0,
        },
        'training_params': {
            'learning_rate': 5e-4,
            'epochs': 50,
            'batch_size': 256,
        }
    }

## Load Datasets

In [ ]:
# Load all available datasets
NN_DATASETS_DIR = BASE_PATH / "nn_datasets"

datasets = {}
for d in [3, 5, 7]:
    path = NN_DATASETS_DIR / f"d{d}_baseline_array.pt"
    if path.exists():
        data = torch.load(path, weights_only=False)
        datasets[d] = {
            'detections': data['detections'].float(),
            'labels': data['labels'].float()
        }
        print(f"d={d}: {len(datasets[d]['labels']):,} samples")
    else:
        print(f"d={d}: NOT FOUND at {path}")

## MixedDataset Implementation

In [ ]:
from benchmark_models import SurfaceCodeSampler

def generate_mixed_batch(ratios: dict, n_samples: int, device: torch.device):
    """
    Generate a mixed batch of training data using Stim.

    Args:
        ratios: Dict {d: proportion}
        n_samples: Total samples to generate
        device: Torch device
    """
    mixed_detections = []
    mixed_labels = []
    mixed_distances = []

    # Calculate counts
    counts = {}
    remaining = n_samples
    keys = sorted(ratios.keys())
    for i, d in enumerate(keys):
        if i == len(keys) - 1:
            count = remaining
        else:
            count = int(n_samples * ratios[d])
            remaining -= count
        counts[d] = count

    # Generate data
    sampler = SurfaceCodeSampler(p=0.001, device=device) # Base P
    p_values = [0.001, 0.003, 0.005, 0.007] # Standard training noise range

    train_groups = []
    for d, count in counts.items():
        if count == 0: continue
        det, lab = sampler.sample(
            d=d,
            num_samples=count,
            p_values=p_values,
            p_weights=[0.25]*4
        )
        train_groups.append((det, lab, d))

    return train_groups


## Define Splits

In [ ]:
# Define training splits
SPLITS = {
    'equal_333333': {3: 0.333, 5: 0.333, 7: 0.334},
    'no_d3': {5: 0.5, 7: 0.5},
    'd5_heavy': {3: 0.2, 5: 0.6, 7: 0.2},
    'd7_only': {7: 1.0},
}

print("Defined splits:")
for name, ratios in SPLITS.items():
    print(f"  {name}: {ratios}")

## Training Loop

In [ ]:
# Training configuration
EPOCHS = 50
BATCH_SIZE = best_config['training_params']['batch_size']
LR = best_config['training_params']['learning_rate']
TOTAL_SAMPLES = 500_000  # Total samples to see over training
REFRESH_EVERY = 10       # Regenerate data every N epochs (match gSAGE)
SAMPLES_PER_BLOCK = 100_000 # Samples per regeneration block
SEED = 42


def train_split(split_name: str, ratios: dict):
    """Train DeepSets on a specific split with Dynamic Regeneration."""
    print(f"\n{'='*60}")
    print(f"Training: {split_name} (Dynamic Generation)")
    print(f"{'='*60}")
    print(f"Ratios: {ratios}")

    model_path = MODELS_DIR / f"{split_name}.pt"
    if model_path.exists():
        print(f"Already trained, skipping.")
        return

    # Initialize model
    model = DeepSets(
        nickname=split_name,
        phi_hidden=tuple(best_config['model_params']['phi_hidden']),
        rho_hidden=tuple(best_config['model_params']['rho_hidden']),
        pool=best_config['model_params']['pool'],
        dropout=0.0,
        device=device,
        base_path=BASE_PATH,
        seed=SEED
    )

    # Setup Optimizer
    model.model.train()
    optimizer = torch.optim.Adam(model.model.parameters(), lr=LR)
    criterion = torch.nn.BCELoss()

    epoch_losses = []

    # Outer loop: Data Refresh Cycles
    num_cycles = EPOCHS // REFRESH_EVERY
    if num_cycles < 1: num_cycles = 1

    for cycle in range(num_cycles):
        print(f"\n[Cycle {cycle+1}/{num_cycles}] Generating new training data ({SAMPLES_PER_BLOCK:,} samples)...")
        train_groups = generate_mixed_batch(ratios, SAMPLES_PER_BLOCK, device)

        # Inner loop: Epochs on this data block
        for cycle_epoch in range(REFRESH_EVERY):
            epoch = cycle * REFRESH_EVERY + cycle_epoch
            if epoch >= EPOCHS: break

            # Create batch list
            all_batches = []
            for det, lab, d in train_groups:
                # Split into mini-batches
                n = len(lab)
                start = 0
                while start < n:
                    end = min(start + BATCH_SIZE, n)
                    all_batches.append((det[start:end], lab[start:end], d))
                    start = end

            random.shuffle(all_batches)

            # Training steps
            total_loss = 0
            n_batches = 0
            running_acc = 0.0

            pbar = tqdm(all_batches, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

            for batch_det, batch_lab, d in pbar:
                batch_det = batch_det.to(device)
                batch_lab = batch_lab.to(device).unsqueeze(-1)

                # Convert
                coords, counts = model._syndromes_to_coords(batch_det, d)

                optimizer.zero_grad()
                pred = model.model(coords, counts)
                loss = criterion(pred, batch_lab)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.model.parameters(), 1.0)
                optimizer.step()

                total_loss += loss.item()
                n_batches += 1
                acc = ((pred > 0.5).float() == batch_lab).float().mean().item()
                running_acc = acc * 0.1 + running_acc * 0.9 if running_acc else acc

                pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{running_acc:.4f}'})

            avg_loss = total_loss / max(n_batches, 1)
            epoch_losses.append(avg_loss)
            print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}, Acc: {running_acc:.4f}")

    # Save model
    torch.save({
        'state_dict': model.model.state_dict(),
        'config': model._config,
        'split_name': split_name,
        'ratios': ratios,
        'epochs': EPOCHS,
        'final_loss': epoch_losses[-1],
        'timestamp': datetime.now().isoformat()
    }, model_path)

    print(f"Model saved to: {model_path}")

    # Save result
    result_path = RESULTS_DIR / f"{split_name}.json"
    with open(result_path, 'w') as f:
        json.dump({
            'split_name': split_name,
            'ratios': ratios,
            'epochs': EPOCHS,
            'final_loss': epoch_losses[-1],
            'timestamp': datetime.now().isoformat()
        }, f, indent=2)


In [ ]:
# Train all splits
for split_name, ratios in SPLITS.items():
    train_split(split_name, ratios)

print("\n" + "="*60)
print("All splits trained!")
print("="*60)